In [1]:
import pyodbc
import pandas as pd

In [2]:
def create_connection():
    # Thông tin kết nối SQL Server
    server = 'ZOHATEA'  # Tên server từ SSMS
    database = 'Customer'  # Đã sửa, bỏ dấu ;
    username = 'sa'
    password = 'ptit'

    # Kết nối SQL Server
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"  # Sử dụng driver có sẵn
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password}"
        )
        cursor = conn.cursor()
        print("Kết nối thành công!")
    except pyodbc.Error as ex:
        print(f"Lỗi kết nối: {ex}")
    return conn, cursor

# 1. Khach Hang

In [3]:
df_khachhang = pd.read_csv('KhachHang.csv', encoding='utf-8')

# lưu vào dim_location
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Location
try:
    cursor.executemany('''
    INSERT INTO KhachHang (MaKH, HoTen, LoaiKhach, QuocTich, Sdt)
    VALUES (?, ?, ?, ?, 0363676283)
    ''', df_khachhang.values.tolist())
    conn.commit()
    print(f"KhachHang: {len(df_khachhang)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
KhachHang: 5000 


# 2. DatPhong

In [4]:
df_datphong = pd.read_csv('DatPhong.csv', encoding='utf-8', keep_default_na=False)
# chi lay hàng co ThoiGian < '2025-06-19'
df_datphong = df_datphong[df_datphong['ThoiGian'] < '2025-06-19']
df_datphong

,MaDatPhong,MaKH,ThoiGian,GhiChu
0,DP00001,CUST02143,2024-01-01,
1,DP00002,CUST04902,2024-01-01,
2,DP00003,CUST03204,2024-01-01,Check-in sớm
3,DP00004,CUST02054,2024-01-01,
4,DP00005,CUST02277,2024-01-01,Check-in sớm
...,...,...,...,...
30160,DP30161,CUST03213,2025-06-18,Check-in sớm
30161,DP30162,CUST00620,2025-06-18,Yêu cầu phòng yên tĩnh
30162,DP30163,CUST02699,2025-06-18,Đặt giúp người khác
30163,DP30164,CUST01641,2025-06-18,Có trẻ nhỏ


In [5]:
# tim max MaDatPhong
max = df_datphong['MaDatPhong'].max()
print(f"Max MaDatPhong: {max}")

Max MaDatPhong: DP30165


In [6]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DatPhong (MaDatPhong, MaKH, ThoiGian, GhiChu)
    VALUES (?, ?, ?, ?)
    ''', df_datphong.values.tolist())
    conn.commit()
    print(f"DatPhong: {len(df_datphong)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DatPhong: 30165 


# 3. PhongDuocDat

In [7]:
df_phongduocdat = pd.read_csv('PhongDuocDat.csv', encoding='utf-8', keep_default_na=False)
df_phongduocdat = df_phongduocdat[df_phongduocdat['MaDatPhong'] <= df_datphong['MaDatPhong'].max()]
df_phongduocdat

,MaPhongDuocDat,MaDatPhong,MaPhong,NgayNhanPhong,NgayTraPhong,TongTienPhong
0,PDD000001,DP00001,THA001_R019,2024-01-01,2024-01-06,8500000
1,PDD000002,DP00002,BRV004_R007,2024-01-01,2024-01-03,2400000
2,PDD000003,DP00003,GLI001_R017,2024-01-01,2024-01-02,1800000
3,PDD000004,DP00003,BDH002_R021,2024-01-01,2024-01-03,4000000
4,PDD000005,DP00004,DLK002_R001,2024-01-01,2024-01-02,1200000
...,...,...,...,...,...,...
40641,PDD040642,DP30163,QTR001_R006,2025-06-18,2025-06-20,2400000
40642,PDD040643,DP30163,BDG001_R012,2025-06-18,2025-06-22,7200000
40643,PDD040644,DP30164,KTM002_R002,2025-06-18,2025-06-20,2400000
40644,PDD040645,DP30165,BRV004_R031,2025-06-18,2025-06-23,11000000


In [8]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO PhongDuocDat (MaPhongDuocDat, MaDatPhong, MaPhong, NgayNhanPhong, NgayTraPhong, TongTienPhong)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_phongduocdat.values.tolist())
    conn.commit()
    print(f"PhongDuocDat: {len(df_phongduocdat)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
PhongDuocDat: 40646 


# 4. DichVu

In [9]:
df_dichvu = pd.read_csv('DichVu.csv', encoding='utf-8', keep_default_na=False)
df_dichvu = df_dichvu[df_dichvu['ThoiGian'] < '2025-06-19']
df_dichvu

,MaSuDungDichVu,MaDichVu,MaPhongDuocDat,ThoiGian,SoLuong,ThanhTien
0,SDV000001,SVC001,PDD000001,2024-01-01,3,900000
1,SDV000002,SVC025,PDD000039,2024-01-01,1,250000
2,SDV000003,SVC010,PDD000039,2024-01-01,2,400000
3,SDV000004,SVC005,PDD000039,2024-01-01,3,2400000
4,SDV000005,SVC006,PDD000039,2024-01-01,2,600000
...,...,...,...,...,...,...
101632,SDV101633,SVC001,PDD040545,2025-06-18,1,300000
101633,SDV101634,SVC020,PDD040533,2025-06-18,1,0
101634,SDV101635,SVC018,PDD040439,2025-06-18,3,6000000
101635,SDV101636,SVC018,PDD040545,2025-06-18,1,2000000


In [10]:
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DichVu (MaDichVuSuDung, MaDichVu, MaPhongDuocDat, ThoiGian, SoLuong, ThanhTien)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_dichvu.values.tolist())
    conn.commit()
    print(f"DichVu: {len(df_dichvu)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DichVu: 101637 


# 5. DanhGia

In [11]:
df_danhgia = pd.read_csv('DanhGia.csv', encoding='utf-8', keep_default_na=False)
df_danhgia = df_danhgia[df_danhgia['ThoiGian'] < '2025-06-19']
df_danhgia

,MaDanhGia,MaPhongDuocDat,ThoiGian,DiemDanhGia,NhanXet
0,DGPDD000005_1,PDD000005,2024-01-03,4,"Tốt, nhưng còn vài điểm cần cải thiện."
1,DGPDD000021_1,PDD000021,2024-01-03,4,"Nhân viên thân thiện, phòng sạch sẽ."
2,DGPDD000062_1,PDD000062,2024-01-03,5,"Rất hài lòng, sẽ quay lại."
3,DGPDD000046_1,PDD000046,2024-01-03,3,"Phục vụ trung bình, giá hợp lý."
4,DGPDD000055_1,PDD000055,2024-01-04,3,"Tạm ổn, cần nâng cấp một số tiện nghi."
...,...,...,...,...,...
36502,DGPDD040424_1,PDD040424,2025-06-18,4,"Tốt, nhưng còn vài điểm cần cải thiện."
36503,DGPDD040213_2,PDD040213,2025-06-18,3,"Phục vụ trung bình, giá hợp lý."
36504,DGPDD040186_1,PDD040186,2025-06-18,4,"Tốt, nhưng còn vài điểm cần cải thiện."
36505,DGPDD040399_1,PDD040399,2025-06-18,5,Tuyệt vời!


In [12]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DanhGia (MaDanhGia, MaPhongDuocDat, ThoiGian, DiemDanhGia, NhanXet)
    VALUES (?, ?, ?, ?, ?)
    ''', df_danhgia.values.tolist())
    conn.commit()
    print(f"DanhGia: {len(df_danhgia)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
DanhGia: 36507 
